In [ ]:
""" Example call for running evaluation
DATASET="path_to/datasets/3faad104-2ab8-4434-816d-474d8d2641db.scb"
DATASET=test
JOB_NAME="cellxgene_3faad1"
LOG_INTERVAL=100
VALID_SIZE_OR_RATIO=0.1
MAX_LENGTH=600
per_proc_batch_size=64
LAYERS=4
MODEL_SCALE=1
python pretrain.py --data-source $DATASET --save-dir ./save/eval-$(date  +%b%d-%H-%M-%Y) --max-seq-len $MAX_LENGTH --batch-size $per_proc_batch_size     --eval-batch-size $(($per_proc_batch_size * 2))     --epochs 3     --lr 0.0001     --warmup-ratio-or-step 0.1     --log-interval $LOG_INTERVAL --trunc-by-sample --no-cls --no-cce --fp16

python pretrain.py \
    --data-source $DATASET \
    --save-dir ./save/eval-$(date +%b%d-%H-%M-%Y) \
    --load-model $CHECKPOINT \
    --max-seq-len $MAX_LENGTH \
    --batch-size $per_proc_batch_size \
    --eval-batch-size $(($per_proc_batch_size * 2)) \
    --epochs 0 \
    --log-interval $LOG_INTERVAL \
    --trunc-by-sample \
    --no-cls \
    --no-cce \
    --fp16

Example call for running fine-tuning
python pretrain.py \
    --data-source $DATASET \
    --save-dir ./save/eval-$(date +%b%d-%H-%M-%Y) \
    --load-model $CHECKPOINT \
    --max-seq-len $MAX_LENGTH \
    --batch-size $per_proc_batch_size \
    --eval-batch-size $(($per_proc_batch_size * 2)) \
    --epochs 3 \
    --lr 0.0001 \
    --warmup-ratio-or-step 0.1 \
    --log-interval $LOG_INTERVAL \
    --trunc-by-sample \
    --no-cls \
    --no-cce \
    --fp16
"""

In [ ]:
# Imports

# %%
import os
import sys
import argparse
import json
import time
from datetime import timedelta
from pathlib import Path
from typing import List, Tuple, Dict, Union, Optional

import scanpy as sc
import numpy as np
import torch
import transformers
from torch import nn
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader, BatchSampler, RandomSampler, SequentialSampler
from datasets import Dataset, load_dataset, concatenate_datasets


sys.path.insert(0, "../")
import scgpt as scg
from scgpt.model import TransformerModel
from scgpt.loss import masked_mse_loss, masked_relative_error
from scgpt.tokenizer import GeneVocab, random_mask_value
from scgpt.scbank import DataBank
from scgpt.utils import MainProcessOnly
from scgpt import logger

In [ ]:
# scanpy settings


sc.set_figure_params(figsize=(4, 4)) # sets scanpy figure size, what figure tho?
sc.settings.verbosity = "debug"
scg.utils.set_seed(42)

In [ ]:
# Arguments

# %%
# argparse
parser = argparse.ArgumentParser()
parser.add_argument(
    "-d",
    "--data-source",
    type=str,
    required=True,
    help='The name of the data source (currently support "scvi" datasets), or the '
    "path to the data file.",
)
parser.add_argument(
    "-s",
    "--save-dir",
    type=str,
    required=True,
    help="The directory to save the trained model and the results.",
)
parser.add_argument(
    "--load-model",
    type=str,
    default=None,
    help="The directory containing the model and configs to load and continue training.",
)

# settings for data
parser.add_argument(
    "--n-hvg",
    type=int,
    default=None,
    help="The number of highly variable genes. If set to 0, will use all genes. "
    "Default is None, which will determine the n_hvg automatically.",
)
parser.add_argument(
    "--valid-size-or-ratio",
    type=float,
    default=0.1,
    help="The ratio or size of the validation set size if split the dataset. "
    "If value is between 0 and 1, will be parsed as the ratio. If value is "
    "greater than 1 and be an integer, will be parsed as the size. If value "
    "is 0, will not split the dataset.",
)

parser.add_argument(
    "--grad-accu-steps",
    type=int,
    default=1,
    help="The number of gradient accumulation steps. Default is 1.",
)

# settings for tokenizer
parser.add_argument(
    "--pad-token",
    type=str,
    default="<pad>",
    help="The token to use for padding. Default is <pad>.",
)
parser.add_argument(
    "--input-style",
    type=str,
    choices=["normed_raw", "log1p", "binned"],
    default="binned",
    help="The style of the input data. Default is binned.",
)
parser.add_argument(
    "--input-emb-style",
    type=str,
    choices=["category", "continuous", "scaling"],
    default="continuous",
    help="The style of the input embedding. Default is continuous.",
)
parser.add_argument(
    "--n-bins",
    type=int,
    default=51,
    help="The number of bins to use for the binned input style. Default is 51.",
)
parser.add_argument(
    "--max-seq-len",
    type=int,
    default=1536,
    help="The maximum length of the sequence. Default is 1000. The actual used "
    "max length would be the minimum of this value and the length of the longest "
    "sequence in the data.",
)
# omit the args for MLM and MVC, will always use them by default
parser.add_argument(
    "--training-tasks",  #  choices of "mlm", "gen", "both"
    type=str,
    default="both",
    choices=["pcpt", "gen", "both"],
    help="The tasks to use for training. pcpt: perception training with maked token "
    "learning. gen: generation. Default is both.",
)
parser.add_argument(
    "--mask-ratio",
    type=float,
    default=0.40,
    help="The ratio of masked values in the training data. Default is 0.40. This"
    "value will be ignored if --training-tasks is set to gen or both.",
)
parser.add_argument(
    "--trunc-by-sample",
    action="store_true",
    help="Whether to truncate the input by sampling rather than cutting off if "
    "sequence length > max_seq_length. Default is False.",
)
parser.add_argument(
    "--vocab-path",
    type=str,
    help="Path to the vocabulary file.",
)
# settings for training
parser.add_argument(
    "--local-rank",
    type=int,
    default=-1,
    help="The local rank of the process for using the torch.distributed.launch "
    "utility. Will be -1 if not running in distributed model.",
)
parser.add_argument(
    "--batch-size",
    type=int,
    default=32,
    help="The batch size for training. Default is 32.",
)
parser.add_argument(
    "--eval-batch-size",
    type=int,
    default=32,
    help="The batch size for evaluation. Default is 32.",
)
parser.add_argument(
    "--epochs",
    type=int,
    default=10,
    help="The number of epochs for training.",
)
parser.add_argument(
    "--lr",
    type=float,
    default=1e-3,
    help="The learning rate for training. Default is 1e-3.",
)
parser.add_argument(
    "--scheduler-interval",
    type=int,
    default=100,
    help="The interval iterations for updating the learning rate. Default is 100. "
    "This will only be used when warmup-ratio is 0.",
)
parser.add_argument(
    "--scheduler-factor",
    type=float,
    default=0.99,
    help="The factor for updating the learning rate. Default is 0.99. "
    "This will only be used when warmup-ratio is 0.",
)
parser.add_argument(
    "--warmup-ratio-or-step",
    type=float,
    default=0.1,
    help="The ratio of warmup steps out of the total training steps. Default is 0.1. "
    "If warmup-ratio is above 0, will use a cosine scheduler with warmup. If "
    "the value is above 1, will use it as the number of warmup steps.",
)
parser.add_argument(
    "--no-cls",
    action="store_true",
    help="Whether to deactivate the classification loss. Default is False.",
)
parser.add_argument(
    "--no-cce",
    action="store_true",
    help="Whether to deactivate the contrastive cell embedding objective. "
    "Default is False.",
)
parser.add_argument(
    "--fp16",
    action="store_true",
    help="Whether to train in automatic mixed precision. Default is False.",
)
parser.add_argument(
    "--fast-transformer",
    type=bool,
    default=True,
    help="Whether to use the fast transformer. Default is True.",
)

# settings for model
parser.add_argument(
    "--nlayers",
    type=int,
    default=4,
    help="The number of layers for the transformer. Default is 4.",
)
parser.add_argument(
    "--nheads",
    type=int,
    default=4,
    help="The number of heads for the transformer. Default is 4.",
)
parser.add_argument(
    "--embsize",
    type=int,
    default=64,
    help="The embedding size for the transformer. Default is 64.",
)
parser.add_argument(
    "--d-hid",
    type=int,
    default=64,
    help="dimension of the feedforward network model in the transformer. "
    "Default is 64.",
)
parser.add_argument(
    "--dropout",
    type=float,
    default=0.2,
    help="The dropout rate. Default is 0.2.",
)
parser.add_argument(
    "--n-layers-cls",
    type=int,
    default=3,
    help="The number of layers for the classification network, including the "
    "output layer. Default is 3.",
)

# settings for logging
parser.add_argument(
    "--log-interval",
    type=int,
    default=100,
    help="The interval for logging. Default is 100.",
)
parser.add_argument(
    "--save-interval",
    type=int,
    default=1000,
    help="The interval for saving the model. Default is 1000.",
)

In [ ]:
# parse arguments, depending on if scg.utils.isnotebook() or not

if scg.utils.isnotebook():
    args = parser.parse_args(
        args=[
            "-d",
            "/scratch/hdd001/home/haotian/datasets/cellxgene/3faad104-2ab8-4434-816d-474d8d2641db.scb",
            "-s",
            "./save/tmp",
            "--batch-size",
            "16",
            "--max-seq-len",
            "512",
            "--trunc-by-sample",
            "--no-cls",
            "--no-cce",
        ]
    )
else:
    args = parser.parse_args()

In [ ]:
# args.local_rank = os.environ['LOCAL_RANK']
# validate settings
assert args.input_style in ["normed_raw", "log1p", "binned"]
assert args.input_emb_style in ["category", "continuous", "scaling"]
if args.input_style == "binned":
    if args.input_emb_style == "scaling":
        raise ValueError("input_emb_style `scaling` is not supported for binned input.")
elif args.input_style == "log1p" or args.input_style == "normed_raw":
    if args.input_emb_style == "category":
        raise ValueError(
            "input_emb_style `category` is not supported for log1p or normed_raw input."
        )

if args.input_emb_style == "category":
    args.mask_value = args.n_bins + 1
    args.pad_value = args.n_bins  # for padding gene expr values
    n_input_bins = args.n_bins + 2
else:
    args.mask_value = -1
    args.pad_value = -2
    n_input_bins = args.n_bins

if args.training_tasks in ["gen", "both"]:
    args.mask_ratio = [0.25, 0.50, 0.75]

In [ ]:
# %% settings
print(args)

special_tokens = [args.pad_token, "<cls>", "<eoc>"]
USE_CLS = not args.no_cls
USE_CCE = not args.no_cce
MVC = True
USE_GENERATIVE_TRAINING = True if args.training_tasks in ["gen", "both"] else False

IS_DATA_PARALLEL = args.local_rank != -1
if IS_DATA_PARALLEL:
    # These two lines is to solve issue #1 based on the suggestion from
    # https://discuss.pytorch.org/t/94382
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.local_rank)

    torch.distributed.init_process_group(
        backend="nccl",
        rank=args.local_rank,
        timeout=timedelta(hours=10),
    )
    # specify device 0 since the CUDA_VISIBLE_DEVICES is set to one GPU
    # https://discuss.pytorch.org/t/67488/4
    device = torch.device("cuda:0")
    n_gpu = torch.cuda.device_count()
    world_size = torch.distributed.get_world_size()
    logger.info(
        f"device: {device} in world size {world_size}, "
        f"visible gpu(s): {os.environ['CUDA_VISIBLE_DEVICES']}/{n_gpu}"
    )
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

save_dir = Path(args.save_dir)
if args.local_rank in [0, -1]:
    save_dir.mkdir(parents=True, exist_ok=True)
    with open(save_dir / "args.json", "w") as f:
        json.dump(vars(args), f, indent=2)
    # copy all uncommitted changes to the save dir
    os.system(
        f"git diff > {str(save_dir / 'git_diff_')}{scg.utils.get_git_commit()}.diff"
    )
if IS_DATA_PARALLEL:
    torch.distributed.barrier()

scg.utils.add_file_handler(logger, save_dir / "run.log")
# log running date and current git commit
logger.info(f"Running on {time.strftime('%Y-%m-%d %H:%M:%S')}")
logger.info(f"Current git commit: {scg.utils.get_git_commit()}")

writer = SummaryWriter(log_dir=save_dir / "tensorboard")
if IS_DATA_PARALLEL:
    writer = MainProcessOnly(writer)